In [0]:
# %sql
# CREATE TABLE IF NOT EXISTS telco_bronze.bronze_dim_tower (
#     tower_id STRING,
#     tower_name STRING,
#     city STRING,
#     state STRING,
#     region STRING,
#     network_type STRING,
#     installation_date DATE,
#     updated_at TIMESTAMP
# ) using delta;

# CREATE TABLE IF NOT EXISTS telco_silver.silver_dim_tower (
#     tower_id STRING,
#     tower_name STRING,
#     city STRING,
#     state STRING,
#     region STRING,
#     network_type STRING,
#     installation_date DATE,
#     updated_at TIMESTAMP
# ) using delta;

# CREATE TABLE IF NOT EXISTS telco_gold.gold_dim_tower_scd1 (
#     tower_id STRING,
#     tower_name STRING,
#     city STRING,
#     state STRING,
#     region STRING,
#     network_type STRING,
#     installation_date DATE,
#     updated_at TIMESTAMP
# ) using delta tblproperties(delta.enableChangeDataFeed = true);

In [0]:
from pyspark.sql.functions import *

container = "source"
storage_account = "sourcesystemadlsgen2"
source_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
schema_location =f"{source_path}/_schemas/tower_schema/"
checkpoint_location = f"{source_path}/_checkpoints/tower_bronze/"
bronze_table = "telco_bronze.bronze_dim_tower"

configs = {
    "cloudFiles.format":"parquet",
    "cloudFiles.inferColumnTypes":"true",
    "cloudFiles.schemaLocation":schema_location,
    "cloudFiles.schemaEvolutionMode":"addNewColumns",
    "cloudFiles.maxFilesPerTrigger":"10"
}

In [0]:
landing_df = spark.readStream.format("cloudFiles")\
    .options(**configs)\
        .load(source_path + "/tower_source/*")

bronze_df = landing_df.withColumn("ingestion_time", current_timestamp())

bronze_query = bronze_df.writeStream.queryName("bronze_ingestion")\
    .format("delta")\
        .outputMode("append")\
            .option("mergeSchema","true")\
                .option("checkpointLocation",checkpoint_location)\
                    .toTable(bronze_table)

In [0]:
%sql
select * from telecom_7405608425992653.telco_bronze.bronze_dim_tower